# Audio Feature Extraction (openSMILE) — IEMOCAP

Extractor: [openSMILE](https://github.com/audeering/opensmile-python) — hand-crafted acoustic functionals  
Feature set: configurable via `FEATURE_SET` in the config cell (default: `eGeMAPSv02`)  
Feature level: `Functionals` — one fixed-size vector per utterance  
Output: `Dataset/Processed/IEMOCAP/features/audio_opensmile_{FEATURE_TAG}.pt`  
Format: `dict {utt_id: np.array(N_FEATURES,)}`  

| Feature set | Dim | Notes |
|---|---|---|
| `eGeMAPSv02` | 88 | Geneva Minimalistic Acoustic Parameter Set v2 — standard for SER |
| `GeMAPSv01b` | 62 | Older GeMAPS baseline |
| `ComParE_2016` | 6373 | Large set used in ComParE challenges |
| `IS09` | 384 | Interspeech 2009 emotion challenge set |
| `IS10` | 1582 | Interspeech 2010 paralinguistics challenge set |
| `IS13` | 6373 | Interspeech 2013 ComParE set |

No GPU needed — CPU-only extraction.

In [1]:
import opensmile
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.notebook import tqdm

In [2]:
PROJECT_ROOT = Path("/mnt/Work/ML/Thesis/BMVC/Hopeful")
DATA_ROOT    = PROJECT_ROOT / "Dataset/Processed/IEMOCAP"
FEATURES_DIR = DATA_ROOT / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Swap feature set here ─────────────────────────────────────────────────────
FEATURE_SET = opensmile.FeatureSet.IS10
# Options:
#   opensmile.FeatureSet.eGeMAPSv02    → 88 features  (recommended for SER)
#   opensmile.FeatureSet.GeMAPSv01b   → 62 features
#   opensmile.FeatureSet.ComParE_2016 → 6373 features
#   opensmile.FeatureSet.IS09         → 384 features  (Interspeech 2009 emotion)
#   opensmile.FeatureSet.IS10         → 1582 features (Interspeech 2010 paralinguistics)
#   opensmile.FeatureSet.IS13         → 6373 features (Interspeech 2013 ComParE)
# ─────────────────────────────────────────────────────────────────────────────
FEATURE_TAG = FEATURE_SET.name  # e.g. "eGeMAPSv02"

print(f"Feature set : {FEATURE_SET.name}")
print(f"Tag         : {FEATURE_TAG}")
print(f"Data root   : {DATA_ROOT}")

Feature set : IS10
Tag         : IS10
Data root   : /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/IEMOCAP


In [3]:
smile = opensmile.Smile(
    feature_set=FEATURE_SET,
    feature_level=opensmile.FeatureLevel.Functionals,
    num_workers=1,
    verbose=False,
)

N_FEATURES = smile.num_features
print(f"Dimensions  : {N_FEATURES}")
print(f"Feature names (first 10): {smile.feature_names[:10]}")

Dimensions  : 1582
Feature names (first 10): ['pcm_loudness_sma_maxPos', 'pcm_loudness_sma_minPos', 'pcm_loudness_sma_amean', 'pcm_loudness_sma_linregc1', 'pcm_loudness_sma_linregc2', 'pcm_loudness_sma_linregerrA', 'pcm_loudness_sma_linregerrQ', 'pcm_loudness_sma_stddev', 'pcm_loudness_sma_skewness', 'pcm_loudness_sma_kurtosis']


In [4]:
labels = pd.read_csv(DATA_ROOT / "labels.csv")
print(f"Utterances : {len(labels)}")
print(f"Columns    : {labels.columns.tolist()}")
labels.head(3)

Utterances : 10039
Columns    : ['utt_id', 'session', 'dialog', 'speaker', 'emotion', 'valence', 'arousal', 'dominance', 'start_time', 'end_time', 'transcription', 'audio_path', 'video_path', 'text_path']


,utt_id,session,dialog,speaker,emotion,valence,arousal,dominance,start_time,end_time,transcription,audio_path,video_path,text_path
0,Ses01F_impro01_F000,Session1,Ses01F_impro01,F,neu,2.5,2.5,2.5,6.2901,8.2357,Excuse me.,audio/Ses01F_impro01_F000.wav,video/Ses01F_impro01_F000.mp4,text/Ses01F_impro01_F000.txt
1,Ses01F_impro01_F001,Session1,Ses01F_impro01,F,neu,2.5,2.5,2.5,10.0100,11.3925,Yeah.,audio/Ses01F_impro01_F001.wav,video/Ses01F_impro01_F001.mp4,text/Ses01F_impro01_F001.txt
2,Ses01F_impro01_F002,Session1,Ses01F_impro01,F,neu,2.5,2.5,2.5,14.8872,18.0175,Is there a problem?,audio/Ses01F_impro01_F002.wav,video/Ses01F_impro01_F002.mp4,text/Ses01F_impro01_F002.txt


In [5]:
def extract_features(path: Path) -> np.ndarray:
    """Extract openSMILE functionals from one audio file → (N_FEATURES,)."""
    df = smile.process_file(str(path))
    return df.values[0].astype(np.float32)  # (N_FEATURES,)

In [6]:
features: dict[str, np.ndarray] = {}
skipped:  list[str] = []

utt_ids     = labels["utt_id"].tolist()
audio_paths = labels["audio_path"].tolist()

for uid, apath in tqdm(zip(utt_ids, audio_paths), total=len(utt_ids), desc="Extracting"):
    p = DATA_ROOT / apath
    if not p.exists():
        skipped.append(uid)
        continue
    try:
        features[uid] = extract_features(p)
    except Exception as e:
        print(f"  [warn] {uid}: {e}")
        skipped.append(uid)

print(f"Extracted : {len(features)}")
print(f"Skipped   : {len(skipped)}  {skipped if skipped else ''}")

Extracting:   0%|          | 0/10039 [00:00<?, ?it/s]

Extracted : 10039
Skipped   : 0  


In [7]:
out_path = FEATURES_DIR / f"audio_opensmile_{FEATURE_TAG}.pt"
torch.save(features, out_path)
size_mb = out_path.stat().st_size / 1e6
print(f"Saved  : {out_path}")
print(f"Size   : {size_mb:.1f} MB")

Saved  : /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/IEMOCAP/features/audio_opensmile_IS10.pt
Size   : 89.8 MB


In [8]:
loaded = torch.load(out_path, weights_only=False)

expected_count = len(labels) - len(skipped)
assert len(loaded) == expected_count, f"Expected {expected_count}, got {len(loaded)}"

sample_key  = next(iter(loaded))
sample_feat = loaded[sample_key]
assert sample_feat.shape == (N_FEATURES,), f"Expected ({N_FEATURES},), got {sample_feat.shape}"

all_feats = np.stack(list(loaded.values()))
has_nan   = np.isnan(all_feats).any()
has_inf   = np.isinf(all_feats).any()

l2_norm   = np.linalg.norm(sample_feat)
sample_row = labels[labels.utt_id == sample_key].iloc[0]

print(f"Count   : {len(loaded)} / {len(labels)} ({len(skipped)} skipped)")
print(f"Shape   : {sample_feat.shape}  ✓")
print(f"Mean    : {all_feats.mean():.4f}")
print(f"Std     : {all_feats.std():.4f}")
print(f"Has NaN : {has_nan}")
print(f"Has Inf : {has_inf}")
print(f"\nSample utt_id : {sample_key}")
print(f"Emotion       : {sample_row.emotion}")
print(f"L2 norm       : {l2_norm:.2f}")

assert not has_nan, "NaN detected"
assert not has_inf, "Inf detected"
print("\nAll assertions passed.")

Count   : 10039 / 10039 (0 skipped)
Shape   : (1582,)  ✓
Mean    : 28.7717
Std     : 230.5400
Has NaN : False
Has Inf : False

Sample utt_id : Ses01F_impro01_F000
Emotion       : neu
L2 norm       : 5146.43

All assertions passed.
